# TBTRL Maze SAC: factorised actor and critic

Both Gaussian mean and log standard deviation use actor state factors and a separate actor goal embedding. The actor objective includes current-task SAC, old-goal policy replay, covariance, goal-separation and soft norm penalties.

Full original Maze SAC task budgets and goals are used by default. The actor settings are an experimental starting point, not tuned performance claims. Logs and plots appear in the training cell after each goal; outputs are also saved.


In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import torch

from tbtrl.experiments.config import load_config
from tbtrl.experiments.diagnostics import NotebookDiagnostics
from tbtrl.experiments.plots import plot_comparison, plot_run
from tbtrl.experiments.runner import run_experiment

root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "configs" / "tbtrl_gridworldmaze_sac_factorised_actor.json").is_file()
)
# Alternatives: "baseline", "factorised_only", "mlp_replay", "factorised_tbtrl".
# Baseline uses the unchanged original MLP actor/config.
VARIANT = "factorised_tbtrl"
if VARIANT == "baseline" or VARIANT == "mlp_replay":
    config = load_config(root / "configs" / "tbtrl_gridworldmaze_sac.json")
    if VARIANT == "mlp_replay":
        config.training["actor_tbtrl"] = {"replay_loss_coef": 1.0}
else:
    config = load_config(root / "configs" / "tbtrl_gridworldmaze_sac_factorised_actor.json")
    if VARIANT == "factorised_only":
        config.training["actor_tbtrl"] = None
    elif VARIANT != "factorised_tbtrl":
        raise ValueError(f"Unknown variant: {VARIANT}")
config.name += f"-{VARIANT}"
# Tune actor loss coefficients independently in config.training["actor_tbtrl"].
SMOKE = False
DEVICE = (
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
if SMOKE:
    config = config.smoke()
output_root = Path(os.environ.get("TBTRL_OUTPUT_ROOT", str(root / "runs")))
output = output_root / f"{config.name}-{uuid4().hex[:8]}"
print("SMOKE CHECK" if SMOKE else "FULL RESEARCH RUN")
print(f"Device: {DEVICE}; seeds: {config.seeds}; goals: {config.goals}")
print(
    f"Per-goal budget: {config.training['total_steps']:,}; warmup: {config.training['warmup_steps']:,}; batch: {config.training['batch_size']}; recovery: {config.recovery['total_steps']:,}"
)
print(
    "Early stopping retains the original threshold and patience; logs print at evaluation intervals."
)
get_ipython().run_line_magic("matplotlib", "inline")
print(
    f"Variant: {VARIANT}; actor: {config.actor_type}; actor TBTRL: {config.training.get('actor_tbtrl')}"
)

In [ ]:
diagnostics = NotebookDiagnostics(rollout_episodes=config.eval_episodes)
records = run_experiment(config, output, device=DEVICE, on_task_end=diagnostics, verbose=True)
print(f"Saved logs, metrics, per-goal diagnostics and checkpoints to {output}")

In [ ]:
figure = plot_run(output)
figure

In [ ]:
comparison = plot_comparison(output)
comparison

Actor factors are private to the actor. Old-task policy terms sample **fresh actions** at replay states and their original goals. Covariance is computed separately for each action component, across states. Alpha uses the same current/old-goal weighting as the actor objective.

The existing critic equations and baseline actor path are preserved. Changing actor architecture also changes initialization and parameter count, so compare multiple seeds and task orders. See `docs/factorised-actor.md` for the equations, options, ablations and validation limits.
